# Оценка численности населения на основе данных реформы ЖКХ


Данные о населении — одни из самых важных в городской аналитике. Однако точной информации нет (возможно ли её получить?), и нам остаётся лишь оценивать. Один из возможных параметров для оценки МКД (многоквартирных домов) — общая жилая площадь. Зная (или рассчитав) среднюю обеспеченность мы можем понять количество человек в каждом доме


#### Реестр многоквартирных домов в рамках реформы ЖКХ

Реестр многоквартирных домов — это система, содержащая информацию о многоквартирных домах в России. Он был создан для улучшения управления жилым фондом, повышения прозрачности в сфере ЖКХ и упрощения контроля за состоянием зданий.

**Что содержится в Реестре МКД:**

- **ID дома** — уникальный идентификатор.
- **Адрес** — регион, город, улица, номер дома.
- **Характеристики дома**:
  - Год постройки, этажность, тип дома.
  - Общая площадь, жилая площадь, нежилая площадь помещений.
  - Состояние здания (аварийное или нет).
  - и другое

Эти данные мы можем так или иначе использовать для оценки численности населения в городах РФ. В отличие от некоторых других источников, реестр уже содержит координаты объектов в полях `lat` и `lon`.


## 0. Импорт библиотек и подготовка данных

### 0.1 Импорт библиотек


In [ ]:
import pandas as pd
import geopandas as gpd

### 0.2 Подготовка данных


В этом разделе будут использованы файлы из нашего репозитория, с первым документом вы уже работали, второй можно скачать по [ссылке](https://drive.google.com/drive/folders/1MkfdDBbyTiZJiztczHOmdTe2aHB-mki4?usp=sharing):

- **spb_admin.gpkg** — полигональные данные о границах районов и округов Санкт-Петербурга.  
  Источник: материалы курса «Методы пространственного анализа», НИУ ВШЭ (Р. Гончаров)

- **spb_mkd_reforma.csv** — данные о многоквартирных домах (МКД) Санкт-Петербурга.  
  Источник: [Фонд развития территорий](https://витрина.фрт.рф/opendata)

Прочитаем данные о границах округов в Санкт-Петербурге (spb_admin.gpkg):


In [ ]:
admin_okrug = gpd.read_file('../data/spb_admin.gpkg', layer='okrug')

admin_okrug.explore(tiles="cartodbpositron")

Прочитаем данные об МКД в Санкт-Петербурге и уберём строки с пустыми координатами:


In [ ]:
mkd = pd.read_csv('../data/spb_mkd_reforma.csv', index_col=0, low_memory=False)
mkd = mkd.dropna(subset=['lon', 'lat'])
mkd.head()

В наборе данных координаты уже записаны в отдельных полях `lat` и `lon`, поэтому можно сразу создать GeoDataFrame:


In [ ]:
mkd_gdf = gpd.GeoDataFrame(mkd, geometry=gpd.points_from_xy(mkd.lon, mkd.lat), crs="EPSG:4326")

## 1. Оценка численности населения в каждом МКД

### 1.1. Проверка системы координат

Перед пространственным объединением проверим, что системы координат у обоих наборов данных совпадают:


In [ ]:
print("CRS точек МКД:", mkd_gdf.crs)
print("CRS округов:", admin_okrug.crs)

### 1.2. Пространственное объединение

Для каждого многоквартирного дома определяем округ, в котором он расположен. Используем метод `sjoin` с предикатом `within`:


In [ ]:
# Пространственное объединение: для каждой точки (дома) находим округ, в который она попадает.
# predicate='within' — точка должна лежать внутри полигона округа.
mkd_okrug = gpd.sjoin(mkd_gdf, admin_okrug, how='left', predicate='within')

# Переименовываем столбцы округа, чтобы с ними было удобнее работать
mkd_okrug = mkd_okrug.rename(columns={
    'NAME': 'okrug_name',        # название округа
    'Popul': 'okrug_population'  # численность населения округа
})

In [ ]:
mkd_okrug.head()

### 1.3. Расчёт обеспеченности жильём и оценка численности населения


Вычислим среднюю обеспеченность жилой площадью на основе данных о населении округа

In [ ]:
# Группируем дома по округу и считаем суммарную площадь жилых помещений в каждом округе
okrug_stat = mkd_okrug.groupby('okrug_name').agg(
    total_residential_area=('area_residential', 'sum')
).reset_index()

okrug_stat.head()

In [ ]:
# Присоединяем суммарную площадь округа к каждому дому по полю 'okrug_name'
mkd_okrug = mkd_okrug.merge(okrug_stat[['okrug_name', 'total_residential_area']], on='okrug_name', how='inner')

# Средняя обеспеченность жилой площадью на одного жителя округа (м²/чел)
mkd_okrug['avg_area_per_person'] = mkd_okrug['total_residential_area'] / mkd_okrug['okrug_population']

mkd_okrug.head()

На основе оценочной обеспеченности жильём вычисляем количество жителей в каждом доме

In [ ]:
# Оцениваем число жителей дома: его жилая площадь / обеспеченность на человека
mkd_okrug['estimated_population'] = mkd_okrug['area_residential'] / mkd_okrug['avg_area_per_person']

mkd_okrug.head()

> **Важно об ограничениях оценки.** Обратите внимание на два момента:
>
> - Мы выводим обеспеченность жильём из **уже известной** численности населения округа (`okrug_population`), а затем распределяем её по домам пропорционально их жилой площади. Это не независимая оценка «с нуля», а **дезагрегация** известного значения: сумма оценок по домам округа примерно равна его исходному населению.
> - Реестр МКД **неполон** — не все дома округа попадают в данные. Там, где домов мало, суммарная жилая площадь занижается, обеспеченность получается нереалистично маленькой (иногда единицы м²/чел), а оценка населения отдельных домов — сильно завышенной (встречаются явно абсурдные значения — тысячи человек в одном доме).
>
> В реальных задачах такие выбросы нужно проверять и отфильтровывать, а полученную обеспеченность — сверять с нормативными значениями (обычно порядка 20–30 м²/чел).

## 2. Создание карты плотности населения


### 2.1. Регулярная сетка

Для визуализации плотности населения создадим регулярную сетку (fishnet) — набор ячеек одинакового размера, покрывающих территорию города:


Создаем функцию построения регулярной сетки для агрегирования данных


In [ ]:
from shapely.geometry import box


def create_regular_grid(data, cell_size):

    # Проверка CRS
    if data.crs is None:
        raise ValueError("У входных данных не задана система координат (CRS).")

    if data.crs.is_geographic:
        data = data.to_crs(data.estimate_utm_crs())

    # Границы области
    minx, miny, maxx, maxy = data.total_bounds

    grid_cells = []
    cell_ids = []

    x = minx
    cell_id = 0  # счётчик ячеек

    while x < maxx:
        y = miny
        while y < maxy:
            grid_cells.append(
                box(x, y, x + cell_size, y + cell_size)
            )
            cell_ids.append(cell_id)
            cell_id += 1
            y += cell_size
        x += cell_size

    # GeoDataFrame
    grid = gpd.GeoDataFrame(
        {
            "cell_id": cell_ids,
            "geometry": grid_cells
        },
        crs=data.crs
    )

    return grid

Создаем сетку для Санкт-Петербурга 1км^2


In [ ]:
grid = create_regular_grid(mkd_okrug, 1000)

Определяем систему координат для перепроецирования данных МКД


In [ ]:
utm_crs = mkd_okrug.estimate_utm_crs()
mkd_okrug_utm = mkd_okrug.to_crs(utm_crs)

### 2.2. Расчёт плотности населения

Агрегируем оценки по ячейкам сетки и вычислим плотность населения (чел./км²) для каждой ячейки:


Рассчитаем плотность населения в каждой ячейке


In [ ]:
mkd_okrug_utm = mkd_okrug_utm.drop(columns=['index_right'])

# Пространственное соединение: сопоставляем каждый дом с ячейкой сетки, в которую он попадает
mkd_in_grid = gpd.sjoin(mkd_okrug_utm, grid, predicate='within')

In [ ]:
# Группируем дома по ячейке сетки (cell_id) и суммируем оценку населения в каждой ячейке
pop_grid = mkd_in_grid.groupby('cell_id')['estimated_population'].sum()

# Превращаем Series в DataFrame с колонками 'cell_id' и 'pop_sum' (суммарное население ячейки)
pop_grid_df = pop_grid.reset_index(name='pop_sum')

# Присоединяем суммы населения к сетке по 'cell_id'
pop_grid_gdf = grid.merge(pop_grid_df, on='cell_id', how='left')

# Плотность населения: человек на квадратный километр (площадь ячейки из м² переводим в км²)
pop_grid_gdf['pop_density'] = pop_grid_gdf['pop_sum'] / (pop_grid_gdf.geometry.area / 1_000_000)

### 2.3. Визуализация


Визуализируем результат


In [ ]:
pop_grid_gdf.explore(column='pop_density', cmap='YlGnBu', tiles='cartodbpositron', scheme='NaturalBreaks', k=5, legend=True, missing_kwds={'color': '#ffffff00','fillOpacity': 0})

## Итог

В этом разделе мы научились оценивать численность населения на основе данных об МКД:

- рассчитали среднюю обеспеченность жилой площадью на жителя по округам;
- оценили численность населения в каждом доме на основе этой обеспеченности;
- агрегировали данные по регулярной сетке и визуализировали пространственное распределение плотности населения.
